# Calculate trades

This notebook shows one practical workflow for building a multi-year MARIO database from repeated parser calls and then using `db.calc_trades(...)` to compare region-by-region trade patterns across scenarios.

It also highlights the newer trade-analysis features exposed by MARIO: Region aggregation through `clusters=...`, directional cluster aggregation with `clusters_direction=...`, split intermediate vs final trade, row and column totals, HTML and image export, custom plot titles, animated multi-scenario plots with `scenario='all'`, and the Chenery-Moses-specific SUT behavior.

The example uses EXIOBASE 3.10.1 monetary IOT files stored locally as yearly zip archives. It is intentionally kept as a workflow example rather than one notebook executed during the documentation build.

In [1]:
import mario

## Define the local EXIOBASE time-series inputs

The new parser-scenario workflow lets one MARIO database instance hold multiple years as separate scenarios. The first parsed year initializes the database, then the remaining years are imported as new scenarios.

In [ ]:
FOLDER = 'YOUR_PATH_HERE' # folder with multiple Exiobase zip archives.
TABLE = 'IOT'
YEARS = range(2013, 2015)
SYSTEM = 'pxp'

# Uncomment and run if you need to download the database.
# You can also download it manually from https://zenodo.org/records/20051562
# mario.download_exiobase(version='3.10.1', system=SYSTEM, years=YEARS, path=FOLDER)

## Initialize the first year and rename the baseline scenario

The first parse creates the database in the standard way. Then `baseline` is renamed to the actual year so that all yearly scenarios follow one consistent naming convention.

In [3]:
db = mario.parse_exiobase(
    path=f'{FOLDER}/{TABLE}_{YEARS[0]}_{SYSTEM}.zip',
    table='IOT',
    unit='Monetary',
)
db.rename_scenario('baseline', YEARS[0])

INFO Parser: reading EXIOBASE IOT from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot__m7qm_iu.
INFO Parser: reading EXIOBASE IOT extensions from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot__m7qm_iu.
INFO Parser: using split extension layout with 7 extension directories.
INFO Parser: EXIOBASE IOT parsed with 200 sectors, 9 value-added rows and 725 extension rows.
INFO Metadata: initialized.


## Import the remaining years as scenarios

Each additional parser call reuses the same `Database` instance and imports one new compatible scenario instead of creating a separate database object. Only the parser raw blocks are imported; derived matrices can still be computed scenario by scenario when needed.

In [4]:
for year in YEARS[1:]:
    print(f'Processing year {year}...')
    db.parse_exiobase(
        path=f'{FOLDER}/{TABLE}_{year}_{SYSTEM}.zip',
        table='IOT',
        unit='Monetary',
        new_scenario=year,
    )
    print('DONE')

Processing year 2014...
INFO Parser: reading EXIOBASE IOT from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_5dkujwse.
INFO Parser: reading EXIOBASE IOT extensions from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_5dkujwse.
INFO Parser: using split extension layout with 7 extension directories.
INFO Parser: EXIOBASE IOT parsed with 200 sectors, 9 value-added rows and 725 extension rows.
INFO Parser: state payload ready with 5 canonical blocks.
DONE


## Calculate and export yearly trade heatmaps for one sector

`db.calc_trades(...)` aggregates one sector or commodity into an origin-by-destination trade matrix, and can also save the corresponding heatmap directly to HTML or image files.

![How calc_trades works](../../_static/images/calc_trades.png)

The next loop calculates one region-by-region trade matrix for `Nickel ores and concentrates` in each scenario, adds totals, and writes the corresponding heatmap to one HTML file per year with a custom title.

## Inspect one scenario with split components and totals

For one scenario, `db.calc_trades(...)` can keep intermediate and final trade separate with `aggregate=False`, and can also append row and column totals with `total=True`. This is useful when you want the raw accounting table instead of only the aggregated heatmap input.

In [5]:
sector = 'Nickel ores and concentrates'
base_scenario = db.scenarios[0]
base_label = str(base_scenario)

In [6]:
split_trade = db.calc_trades(
    item=sector,
    scenario=base_scenario,
    aggregate=False,
    total=True,
    auto_open=False,
    path=False,
    show_plot=False,
)
split_trade

Intermediate                                                            \
Region           AT           AU        BE   BG          BR           CA   CH   
Region                                                                          
AT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
AU              0.0  2096.333717  0.008412  0.0   42.148222    71.910781  0.0   
BE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
BG              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
BR              0.0     0.000000  0.087826  0.0  576.120518     0.118718  0.0   
CA              0.0     0.000000  0.000000  0.0    0.000000  1105.895311  0.0   
CH              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CN              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CY              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CZ              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
DE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
DK              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
EE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
ES              0.0     0.000000  0.000000  0.0    0.000000     0.001242  0.0   
FI              0.0     0.000000  0.000000  0.0    0.000000    51.907989  0.0   
FR              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
GB              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
GR              0.0     0.000000  0.908098  0.0    0.000000     0.000000  0.0   
HR              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
HU              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
ID              0.0    13.961305  0.000000  0.0    0.000000     0.000000  0.0   
IE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
IN              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
IT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
JP              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
KR              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
LT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
LU              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
LV              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
MT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
MX              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
NL              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
NO              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
PL              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
PT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
RO              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
RU              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
SE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
SI              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
SK              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
TR              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
TW              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
US              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
WA              0.0    56.289255  0.018823  0.0    0.000000     0.000000  0.0   
WE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
WF              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
WL              0.0     0.000000  0.00

## Aggregate origin, destination, or both Region axes

`db.calc_trades(...)` can also aggregate Regions before plotting or summing totals. `clusters_direction='origin'` aggregates only the origin Regions, `destination` aggregates only the destination Regions, and `both` aggregates the full square matrix. The next example rolls the origin side of the EXIOBASE trade table up to continent groups.

In [7]:
origin_clustered_trades = db.calc_trades(
    item=sector,
    scenario=base_scenario,
    clusters='continent',
    clusters_direction='origin',
    total=True,
    auto_open=False,
    path=False,
    show_plot=False,
)
origin_clustered_trades

Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,TR,TW,US,WA,WE,WF,WL,WM,ZA,Total
Region,,,,,,,,,,,,,,,,,,,,,
Europe,0.0,0.000000,0.908103,0.0,0.000000,51.909231,0.0,160.407913,9.757916e-13,1.659926,...,0.002653,0.0,0.002786,0.072842,132.738378,15.831240,0.000000,0.000000,4.469047,2236.514003
Asia,0.0,70.382138,0.018823,0.0,0.000000,0.000000,0.0,5673.822022,0.000000e+00,0.000000,...,26.589091,0.0,71.057130,841.185627,52.852502,0.003665,4.146893,236.196678,0.000000,9182.657505
America,0.0,0.000000,0.087826,0.0,592.822909,1106.740844,0.0,5.881573,0.000000e+00,0.000000,...,0.000000,0.0,0.006253,0.000000,56.187196,0.000000,318.642076,2.372726,0.000000,2156.621242
Oceania,0.0,2097.503758,0.008412,0.0,55.834461,71.910781,0.0,1378.830230,0.000000e+00,0.000000,...,0.000000,0.0,0.000000,0.015612,0.000000,0.000000,0.000000,0.948428,0.000000,3639.674247
Africa,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,21.008060,0.000000e+00,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,1089.456976,0.000000,0.000000,448.475774,1581.516306
Total,0.0,2167.885896,1.023165,0.0,648.657370,1230.560856,0.0,7239.949798,9.757916e-13,1.659926,...,26.591744,0.0,71.066169,841.274081,241.778076,1105.291881,322.788969,239.517833,452.944821,18796.983303


## Compare all scenarios with one animated plot

When more than one scenario is available, `scenario='all'` returns one trade matrix per scenario and, with `show_plot=True`, builds one animated heatmap with a scenario slider.

In [8]:
all_trades = db.calc_trades(
    item=sector,
    scenario='all',
    show_plot=True,
    auto_open=False,
    path=False,
    title=f'{sector} - all scenarios',
)

The same `save_plot` argument also supports image extensions such as `.png` or `.svg`. When a path is provided, MARIO saves the figure and still returns the trade matrix without opening or displaying the plot.

## Visual comparison: 2013 vs 2014

The two static heatmaps below were exported from the same workflow for 2013 and 2014. They illustrate one substantive supply-chain shift: the Indonesia-to-China flow of nickel ores becomes much less prominent after 2013, consistent with Indonesia's policy move to stop exporting raw nickel ores.

### 2013

![Nickel ores trade heatmap for 2013](../../_static/images/supply_chain_trades_2013.png)

### 2014

![Nickel ores trade heatmap for 2014](../../_static/images/supply_chain_trades_2014.png)

In this pair of plots, the `ID -> CN` cell is much more visible in 2013 than in 2014. Even in this simple visual form, `db.calc_trades(...)` is already useful for tracking discontinuities in supply-chain geography before moving to more structured indicators.